# Gradient Boosting Intuition & Theory

In this notebook, we'll demonstrate the core mechanism of Gradient Boosting Machines (GBM).
Instead of trying to predict the target $Y$ directly, each new tree in a GBM tries to predict the **errors** (residuals) of the combined previous trees.

## 1. Why is it called "Gradient" Boosting?

The intuition of "predicting the residuals" is actually a mathematical consequence of performing **Gradient Descent in Function Space**.

Let our loss function be the Mean Squared Error (MSE):
$$ L(y, \hat{y}) = \frac{1}{2} (y - \hat{y})^2 $$

If we want to update our prediction $\hat{y}$ to minimize this loss using gradient descent, we take the negative derivative (gradient) of the loss with respect to our prediction $\hat{y}$:

$$ - \frac{\partial L(y, \hat{y})}{\partial \hat{y}} = - \frac{\partial}{\partial \hat{y}} \left[ \frac{1}{2} (y - \hat{y})^2 \right] = - (y - \hat{y}) \cdot (-1) = y - \hat{y} $$

**The negative gradient is exactly equal to the residual $y - \hat{y}$ !**

So, by training a new tree to predict the residual $y - \hat{y}$, we are mathematically training a tree to predict the *negative gradient* of the MSE loss function. When we add this tree's prediction to our existing model, we are taking a "step" in the direction that minimizes the loss.

Let's build a GBM from scratch using our custom `DecisionTreeRegressorFromScratch`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# ---------------------------------------------------------
# Paste our custom Tree Regressor from the previous notebook
# ---------------------------------------------------------
class Node:
    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
    def is_leaf_node(self):
        return self.value is not None

class DecisionTreeRegressorFromScratch:
    def __init__(self, max_depth=2):
        self.max_depth = max_depth
        self.root = None
        
    def fit(self, X, y):
        self.root = self._grow_tree(X, y, 0)
        
    def _grow_tree(self, X, y, depth):
        n_samples = X.shape[0]
        if depth >= self.max_depth or n_samples < 2 or len(np.unique(y)) == 1:
            return Node(value=np.mean(y))
        
        best_feat, best_thresh = self._best_split(X, y)
        if best_feat is None:
            return Node(value=np.mean(y))
            
        left_idxs = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idxs = np.where(X[:, best_feat] > best_thresh)[0]
        
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)
        
    def _best_split(self, X, y):
        best_mse = float('inf')
        best_feat, best_thresh = None, None
        for feat_idx in range(X.shape[1]):
            X_column = X[:, feat_idx]
            for thresh in np.unique(X_column):
                left_idxs = np.where(X_column <= thresh)[0]
                right_idxs = np.where(X_column > thresh)[0]
                if len(left_idxs) == 0 or len(right_idxs) == 0: continue
                mse = self._calculate_split_mse(y, left_idxs, right_idxs)
                if mse < best_mse:
                    best_mse, best_feat, best_thresh = mse, feat_idx, thresh
        return best_feat, best_thresh
        
    def _calculate_split_mse(self, y, left_idxs, right_idxs):
        y_l, y_r = y[left_idxs], y[right_idxs]
        var_l = np.var(y_l) if len(y_l) > 0 else 0
        var_r = np.var(y_r) if len(y_r) > 0 else 0
        return (len(y_l)/len(y)) * var_l + (len(y_r)/len(y)) * var_r
        
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])
        
    def _traverse_tree(self, x, node):
        if node.is_leaf_node(): return node.value
        if x[node.feature_idx] <= node.threshold: return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

## 2. Create a Toy Dataset

In [ ]:
X = np.sort(5 * np.random.rand(80, 1), axis=0)
y = np.sin(X).ravel() + np.random.normal(0, 0.1, X.shape[0])

plt.scatter(X, y, color="darkorange", label="Data")
plt.title("Target Data")
plt.legend()
plt.show()

## 3. Step-by-Step Gradient Boosting
### Iteration 0: The Baseline
We start by predicting the mean of $y$ for all samples (the prediction that minimizes MSE globally).

In [ ]:
F0 = np.mean(y)
predictions = np.full_like(y, F0)

plt.scatter(X, y, color="darkorange", label="Data")
plt.plot(X, predictions, color="navy", label="F0 (Mean)", linewidth=2)
plt.title("Iteration 0: Baseline Prediction")
plt.legend()
plt.show()

### Iteration 1
1. Calculate the negative gradient (residuals): $r_1 = y - \text{predictions}$
2. Train a weak tree (`tree1`, depth 1) to predict these residuals.
3. Update our predictions: $\text{predictions} = \text{predictions} + \eta \times \text{tree1.predict}(X)$

In [ ]:
learning_rate = 0.5

# 1. Calculate residuals (negative gradient)
r1 = y - predictions

# 2. Fit tree to residuals
tree1 = DecisionTreeRegressorFromScratch(max_depth=1)
tree1.fit(X, r1)

# 3. Update predictions
predictions = predictions + learning_rate * tree1.predict(X)

# Plotting the new prediction
plt.scatter(X, y, color="darkorange", label="Data")
plt.plot(X, predictions, color="navy", label="F1 (F0 + tree1)", linewidth=2)
plt.title("Iteration 1")
plt.legend()
plt.show()

## 4. The Impact of Learning Rate (Shrinkage)
Let's put this into a loop and run it for 50 iterations. We will compare a model with a high learning rate ($\eta=1.0$) which is prone to overfitting, versus a model with a lower learning rate (shrinkage, $\eta=0.1$).

In [ ]:
def train_gbm(X, y, n_estimators, learning_rate, max_depth=2):
    preds = np.full_like(y, np.mean(y))
    for _ in range(n_estimators):
        residuals = y - preds
        tree = DecisionTreeRegressorFromScratch(max_depth=max_depth)
        tree.fit(X, residuals)
        preds += learning_rate * tree.predict(X)
    return preds

# Train two models
preds_high_lr = train_gbm(X, y, n_estimators=50, learning_rate=1.0)
preds_low_lr = train_gbm(X, y, n_estimators=50, learning_rate=0.1)

# Plotting
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X, y, color="darkorange", label="Data", alpha=0.6)
plt.plot(X, preds_high_lr, color="firebrick", label="GBM (LR=1.0)", linewidth=2)
plt.title("Overfitting with High Learning Rate (No Shrinkage)")
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(X, y, color="darkorange", label="Data", alpha=0.6)
plt.plot(X, preds_low_lr, color="navy", label="GBM (LR=0.1)", linewidth=2)
plt.title("Smoother Fit with Low Learning Rate (Shrinkage)")
plt.legend()

plt.tight_layout()
plt.show()

## Conclusion
You have just manually built a Gradient Boosting Machine entirely from scratch!

**Why XGBoost?**
While the concept above is powerful, it is computationally slow on large datasets and our simple negative gradient derivation only applies easily to MSE loss.

XGBoost optimizes this exact process by:
1. **Second-Order Taylor Expansion**: Instead of just using the first derivative (gradient), it uses both the first and second derivatives (Hessians) to allow for any arbitrary, twice-differentiable custom loss function.
2. **Regularization**: Adding mathematical penalties for tree complexity to prevent the overfitting seen in our high learning-rate example.
3. **Systems Optimization**: Cache-aware access, parallelized tree building, and handling out-of-core data.